# CSV peek

A quick, nice-looking look at the top of a CSV.

**How to use:** run the setup cell once, then set `CSV_PATH` in the last cell and run it. Re-run the last cell for any other file.

It prints the file size and shape, a per-column summary (dtype, null count, an example value), then the first `n` rows as a table with long text trimmed. Only the first 100 rows are read for speed, so on a large file the null counts are labelled as coming from that sample; on a file that fits, they are exact. The delimiter is sniffed automatically, or pass `sep=';'`.

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ---- config ----
CELL_WIDTH = 60          # long text cells are trimmed to this many characters
STATS_ROWS = 100         # rows read to build the summary; keeps big files fast


def _human_size(num_bytes):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num_bytes < 1024 or unit == "TB":
            return f"{num_bytes:.0f} {unit}" if unit == "B" else f"{num_bytes:.1f} {unit}"
        num_bytes /= 1024


def _trim(v, width=CELL_WIDTH):
    t = "" if v is None else str(v)
    t = t.replace("\n", " ").replace("\r", " ")
    return t[: width - 3] + "..." if len(t) > width else t


def _count_rows(path):
    total = 0
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            total += chunk.count(b"\n")
    return max(total - 1, 0)


def peek(path, n=10, sep=None):
    """Show a nice preview of the top of a CSV. Returns the head DataFrame.

    path : path to the CSV
    n    : number of rows to show (default 10)
    sep  : column separator; sniffed automatically when left as None
    """
    read_rows = max(n, STATS_ROWS)
    if sep is None:
        df = pd.read_csv(path, sep=None, engine="python", nrows=read_rows)  # sniff delimiter
    else:
        df = pd.read_csv(path, sep=sep, nrows=read_rows)

    if len(df) < read_rows:                 # short read means we saw the whole file
        total, exact = len(df), True
    else:
        total, exact = _count_rows(path), False

    summary = pd.DataFrame({
        "column":  [str(c) for c in df.columns],
        "dtype":   [str(t) for t in df.dtypes],
        "nulls":   df.isna().sum().to_numpy(),
        "example": [_trim(df[c].dropna().iloc[0]) if df[c].notna().any() else "" for c in df.columns],
    })

    scope = "" if exact else f"  (summary from first {len(df):,} rows)"
    header = f"{os.path.basename(path)}  |  {_human_size(os.path.getsize(path))}  |  {total:,} rows x {df.shape[1]} columns{scope}"
    print(header)
    display(summary)

    head = df.head(n).copy()
    for c in head.columns:
        if head[c].dtype == "object" or pd.api.types.is_string_dtype(head[c]):
            head[c] = head[c].map(_trim)
    try:
        display(head.style.set_caption(f"top {n} rows").format(precision=2))
    except Exception:
        display(head)
    return df.head(n)


In [ ]:
CSV_PATH = "data.csv"   # <- point this at your file

df_head = peek(CSV_PATH, n=10)
# peek(CSV_PATH, n=20)          # show more rows
# peek(CSV_PATH, sep=";")       # force a delimiter
# df_head                        # the returned frame, if you want to keep working with it